## Group 19 Visualizations


In [1]:
# ------------------------------------------------------------
# Melbourne Nightclubbing Data Story
# ------------------------------------------------------------
# Goal:
# Find the best clubbing areas in Melbourne by combining:
#   - weekend late-night pedestrian activity
#   - hourly nightlife patterns
#   - weekday/weekend differences
#   - seasonal trends
#   - nearby club review popularity
#   - nearby club ratings
#
# Website outputs:
#   - PNG charts
#   - CSV tables
#   - Folium HTML maps
#   - optional Plotly interactive HTML charts
# ------------------------------------------------------------

from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import folium
from folium.plugins import HeatMap, HeatMapWithTime, MarkerCluster

warnings.filterwarnings("ignore")

# Optional interactive website charts
try:
    import plotly.express as px
    import plotly.graph_objects as go
    PLOTLY_AVAILABLE = True
except ImportError:
    PLOTLY_AVAILABLE = False
    print("Plotly not installed. Matplotlib and Folium outputs will still work.")


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

RAW_DIR = Path("../data/raw")
OUT_DIR = Path("../outputs/nightclubbing_story")
MAP_DIR = OUT_DIR / "maps"
CHART_DIR = OUT_DIR / "charts"
TABLE_DIR = OUT_DIR / "tables"
HTML_DIR = OUT_DIR / "html"
TEXT_DIR = OUT_DIR / "text"

for folder in [OUT_DIR, MAP_DIR, CHART_DIR, TABLE_DIR, HTML_DIR, TEXT_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

counts_path = RAW_DIR / "pedestrian-counting-system-monthly-counts-per-hour.csv"
loc_path = RAW_DIR / "pedestrian-counting-system-sensor-locations.csv"

print("Output folder:", OUT_DIR.resolve())

Output folder: C:\Users\jakok\VS_projects\Jakob-kild.github.io\outputs\nightclubbing_story


In [2]:
# ------------------------------------------------------------
# Load raw pedestrian count and sensor location data
# ------------------------------------------------------------

df_counts_raw = pd.read_csv(counts_path)
df_loc_raw = pd.read_csv(loc_path)


def clean_columns(dataframe):
    dataframe = dataframe.copy()
    dataframe.columns = (
        dataframe.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_", regex=False)
    )
    return dataframe


df_counts = clean_columns(df_counts_raw)
df_loc = clean_columns(df_loc_raw)


# Standardise count columns
df_counts = df_counts.rename(columns={
    "location_id": "sensor_id",
    "sensing_date": "date",
    "hourday": "hour",
    "total_of_directions": "pedestrian_count",
    "sensor_name": "sensor_name_counts"
})

# Standardise location columns
df_loc = df_loc.rename(columns={
    "location_id": "sensor_id",
    "sensor_name": "sensor_name_location",
    "sensor_description": "sensor_description"
})


# Parse types
df_counts["date"] = pd.to_datetime(df_counts["date"], errors="coerce")
df_counts["hour"] = pd.to_numeric(df_counts["hour"], errors="coerce")
df_counts["pedestrian_count"] = pd.to_numeric(df_counts["pedestrian_count"], errors="coerce")


# Parse coordinates from the count file
# Example location string: "-37.81295822, 144.95678789"
coords = df_counts["location"].astype(str).str.split(",", expand=True)

df_counts["latitude_from_counts"] = pd.to_numeric(coords[0].str.strip(), errors="coerce")
df_counts["longitude_from_counts"] = pd.to_numeric(coords[1].str.strip(), errors="coerce")


# Merge count data with sensor metadata
df = df_counts.merge(
    df_loc[
        [
            "sensor_id",
            "sensor_description",
            "sensor_name_location",
            "installation_date",
            "note",
            "location_type",
            "status",
            "direction_1",
            "direction_2",
            "latitude",
            "longitude"
        ]
    ],
    on="sensor_id",
    how="left"
)


# Use coordinates from counts first because they better reflect historical sensor position
df["lat"] = df["latitude_from_counts"].fillna(df["latitude"])
df["lon"] = df["longitude_from_counts"].fillna(df["longitude"])

df["sensor_name"] = df["sensor_name_counts"].fillna(df["sensor_name_location"])

df["sensor_label"] = (
    df["sensor_description"]
    .fillna(df["sensor_name"])
    .fillna("Sensor " + df["sensor_id"].astype(str))
)


# Remove unusable rows
df = df.dropna(subset=["date", "hour", "pedestrian_count", "lat", "lon"]).copy()

df["sensor_id"] = df["sensor_id"].astype(int)
df["hour"] = df["hour"].astype(int)
df["pedestrian_count"] = df["pedestrian_count"].clip(lower=0)

print("Rows:", len(df))
print("Sensors:", df["sensor_id"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())

df.head()

Rows: 1557320
Sensors: 100
Date range: 2024-05-08 00:00:00 to 2026-05-07 00:00:00


,id,sensor_id,date,hour,direction_1_x,direction_2_x,pedestrian_count,sensor_name_counts,location,latitude_from_counts,...,location_type,status,direction_1_y,direction_2_y,latitude,longitude,lat,lon,sensor_name,sensor_label
0,1081220240830,108,2024-08-30,12,112,282,394,261Will_T,"-37.81295822, 144.95678789",-37.812958,...,Outdoor,A,North,South,-37.812958,144.956788,-37.812958,144.956788,261Will_T,William St - Little Lonsdale St (West)
1,1081220240830,108,2024-08-30,12,112,282,394,261Will_T,"-37.81295822, 144.95678789",-37.812958,...,Outdoor,A,In,Out,-37.812958,144.956788,-37.812958,144.956788,261Will_T,William St - Little Lonsdale St (West)
2,1081220240830,108,2024-08-30,12,112,282,394,261Will_T,"-37.81295822, 144.95678789",-37.812958,...,Outdoor,A,North,InOut,-37.812958,144.956788,-37.812958,144.956788,261Will_T,William St - Little Lonsdale St (West)
3,107020240528,107,2024-05-28,0,16,10,26,280Will_T,"-37.81246271, 144.95690188",-37.812463,...,Outdoor,A,North,South,-37.812463,144.956902,-37.812463,144.956902,280Will_T,Flagstaff station (East)
4,165920250505,165,2025-05-05,9,21,46,67,Spen475_T,"-37.80953359, 144.94939004",-37.809534,...,Outdoor,A,North,South,-37.809534,144.949390,-37.809534,144.949390,Spen475_T,475 Spencer Street


In [3]:
# ------------------------------------------------------------
# Nightlife logic
# ------------------------------------------------------------
# Important:
# 00:00-05:00 belongs to the previous nightlife date.
# Example:
#   Saturday 01:00 is treated as Friday night.
# This makes more sense for clubbing analysis than calendar date.

df["nightlife_date"] = df["date"] - pd.to_timedelta((df["hour"] < 6).astype(int), unit="D")

df["calendar_weekday_num"] = df["date"].dt.dayofweek
df["calendar_weekday_name"] = df["date"].dt.day_name()

df["nightlife_weekday_num"] = df["nightlife_date"].dt.dayofweek
df["nightlife_weekday_name"] = df["nightlife_date"].dt.day_name()

# Friday and Saturday nights are treated as weekend nightlife
df["nightlife_day_type"] = np.where(
    df["nightlife_weekday_num"].isin([4, 5]),
    "Weekend night",
    "Weekday night"
)


def classify_period(hour):
    if 18 <= hour <= 21:
        return "Evening, 18:00-21:59"
    elif hour >= 22 or hour <= 2:
        return "Late night, 22:00-02:59"
    elif 3 <= hour <= 5:
        return "Overnight, 03:00-05:59"
    elif 6 <= hour <= 11:
        return "Morning, 06:00-11:59"
    elif 12 <= hour <= 17:
        return "Afternoon, 12:00-17:59"
    else:
        return "Other"


df["period"] = df["hour"].apply(classify_period)

df["is_evening"] = df["period"].eq("Evening, 18:00-21:59")
df["is_late_night"] = df["period"].eq("Late night, 22:00-02:59")
df["is_overnight"] = df["period"].eq("Overnight, 03:00-05:59")

df["is_nightlife_hour"] = df["period"].isin([
    "Evening, 18:00-21:59",
    "Late night, 22:00-02:59",
    "Overnight, 03:00-05:59"
])

df["is_weekend_night"] = df["nightlife_day_type"].eq("Weekend night")
df["is_weekday_night"] = df["nightlife_day_type"].eq("Weekday night")


# ------------------------------------------------------------
# Australian seasons
# ------------------------------------------------------------

def australian_season(month):
    if month in [12, 1, 2]:
        return "Summer"
    elif month in [3, 4, 5]:
        return "Autumn"
    elif month in [6, 7, 8]:
        return "Winter"
    elif month in [9, 10, 11]:
        return "Spring"


df["season"] = df["nightlife_date"].dt.month.apply(australian_season)

# Season year:
# Dec 2023, Jan 2024, Feb 2024 are all Summer 2024
df["season_year"] = df["nightlife_date"].dt.year

df.loc[df["nightlife_date"].dt.month == 12, "season_year"] += 1

season_order = ["Summer", "Autumn", "Winter", "Spring"]
weekday_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
nightlife_hour_order = [18, 19, 20, 21, 22, 23, 0, 1, 2, 3, 4, 5]

df["season"] = pd.Categorical(df["season"], categories=season_order, ordered=True)
df["nightlife_weekday_name"] = pd.Categorical(
    df["nightlife_weekday_name"],
    categories=weekday_order,
    ordered=True
)

df[[
    "date",
    "hour",
    "nightlife_date",
    "nightlife_weekday_name",
    "nightlife_day_type",
    "period",
    "season",
    "season_year"
]].head()

,date,hour,nightlife_date,nightlife_weekday_name,nightlife_day_type,period,season,season_year
0,2024-08-30,12,2024-08-30,Friday,Weekend night,"Afternoon, 12:00-17:59",Winter,2024
1,2024-08-30,12,2024-08-30,Friday,Weekend night,"Afternoon, 12:00-17:59",Winter,2024
2,2024-08-30,12,2024-08-30,Friday,Weekend night,"Afternoon, 12:00-17:59",Winter,2024
3,2024-05-28,0,2024-05-27,Monday,Weekday night,"Late night, 22:00-02:59",Autumn,2024
4,2025-05-05,9,2025-05-05,Monday,Weekday night,"Morning, 06:00-11:59",Autumn,2025


In [5]:
# ------------------------------------------------------------
# Club research
# ------------------------------------------------------------
# Review count is treated as the main popularity signal.
# Rating is treated as a secondary quality signal.
# Sensors are treated as proxies for nearby street activity.

club_rows = [
    {
        "club_name": "Melbourne Club",
        "rating": 4.5,
        "review_count": 53,
        "closest_sensor_ids": [18],
        "closest_sensor_description_manual": "Collins Place North",
    },
    {
        "club_name": "Athenaeum Club",
        "rating": 4.6,
        "review_count": 71,
        "closest_sensor_ids": [39],
        "closest_sensor_description_manual": "Alfred Place",
    },
    {
        "club_name": "Alexandra Club",
        "rating": 4.8,
        "review_count": 17,
        "closest_sensor_ids": [39],
        "closest_sensor_description_manual": "Alfred Place",
    },
    {
        "club_name": "Kelvin Club",
        "rating": 4.4,
        "review_count": 142,
        "closest_sensor_ids": [39],
        "closest_sensor_description_manual": "Alfred Place",
    },
    {
        "club_name": "The Charlton Club",
        "rating": 4.0,
        "review_count": 2455,
        "closest_sensor_ids": [21, 63],
        "closest_sensor_description_manual": "155-161 Russell Street; 231 Bourke Street",
    },
    {
        "club_name": "Sub Club Melbourne",
        "rating": 3.8,
        "review_count": 171,
        "closest_sensor_ids": [84],
        "closest_sensor_description_manual": "Elizabeth Street - Flinders Street East",
    },
    {
        "club_name": "Mendoza Social Club",
        "rating": 4.1,
        "review_count": 175,
        "closest_sensor_ids": [84],
        "closest_sensor_description_manual": "Elizabeth Street - Flinders Street East",
    },
    {
        "club_name": "Club Moda",
        "rating": 4.6,
        "review_count": 42,
        "closest_sensor_ids": [36],
        "closest_sensor_description_manual": "Queen Street West",
    },
    {
        "club_name": "Club W",
        "rating": 2.6,
        "review_count": 169,
        "closest_sensor_ids": [52, 56],
        "closest_sensor_description_manual": "Elizabeth Street - Lonsdale Street",
    },
    {
        "club_name": "Club Retro",
        "rating": 3.7,
        "review_count": 1201,
        "closest_sensor_ids": [52, 56],
        "closest_sensor_description_manual": "Elizabeth Street - Lonsdale Street",
    },
    {
        "club_name": "Emo Never Sleeps",
        "rating": 4.8,
        "review_count": 36,
        "closest_sensor_ids": [182],
        "closest_sensor_description_manual": "163 King Street",
    },
    {
        "club_name": "Mango Club",
        "rating": 4.5,
        "review_count": 910,
        "closest_sensor_ids": [182],
        "closest_sensor_description_manual": "163 King Street",
    },
    {
        "club_name": "Bar 20 Sportsbar",
        "rating": 4.6,
        "review_count": 428,
        "closest_sensor_ids": [182],
        "closest_sensor_description_manual": "163 King Street",
    },
    {
        "club_name": "Levels Melbourne",
        "rating": 4.2,
        "review_count": 776,
        "closest_sensor_ids": [182],
        "closest_sensor_description_manual": "163 King Street",
    },
    {
        "club_name": "Decibles Nightclub",
        "rating": 2.8,
        "review_count": 60,
        "closest_sensor_ids": [131],
        "closest_sensor_description_manual": "I-Hub corner of King Street and Flinders Street",
    },
    {
        "club_name": "District 1 Melbourne",
        "rating": 4.2,
        "review_count": 558,
        "closest_sensor_ids": [184],
        "closest_sensor_description_manual": "124 Elizabeth Street",
    },
]

df_clubs = pd.DataFrame(club_rows)

df_clubs["log_reviews"] = np.log1p(df_clubs["review_count"])
df_clubs["rating_x_log_reviews"] = df_clubs["rating"] * df_clubs["log_reviews"]

club_sensor_pairs = (
    df_clubs
    .explode("closest_sensor_ids")
    .rename(columns={"closest_sensor_ids": "sensor_id"})
    .copy()
)

club_sensor_pairs["sensor_id"] = club_sensor_pairs["sensor_id"].astype(int)


# ------------------------------------------------------------
# Remove duplicate / unwanted venue-to-sensor links
# ------------------------------------------------------------
# Important:
# Do this BEFORE club_sensor_ids, club_sensor_summary, story_places, and charts
# are created. If you only remove these links later in a map cell, the charts
# will still contain duplicate club labels because the summaries have already
# been built.
#
# These choices keep one preferred proxy sensor per club:
#   - The Charlton Club -> sensor 63, not sensor 21
#   - Club W -> sensor 56, not sensor 52
#   - Club Retro -> sensor 52, not sensor 56

club_sensor_pairs["club_name_clean"] = club_sensor_pairs["club_name"].str.strip().str.lower()

links_to_remove = [
    ("club w", 52),
    ("club retro", 56),
    ("the charlton club", 21),
]

remove_mask = pd.Series(False, index=club_sensor_pairs.index)

for club_name, sensor_id in links_to_remove:
    remove_mask = remove_mask | (
        (club_sensor_pairs["club_name_clean"] == club_name) &
        (club_sensor_pairs["sensor_id"] == sensor_id)
    )

removed_links = club_sensor_pairs.loc[
    remove_mask,
    ["club_name", "sensor_id", "closest_sensor_description_manual"]
].copy()

club_sensor_pairs = (
    club_sensor_pairs
    .loc[~remove_mask]
    .drop(columns=["club_name_clean"])
    .copy()
)


club_sensor_ids = sorted(club_sensor_pairs["sensor_id"].unique())

print("Clubs:", df_clubs["club_name"].nunique())
print("Club-sensor pairs after cleanup:", len(club_sensor_pairs))
print("Unique nearby sensors after cleanup:", len(club_sensor_ids))

club_sensor_pairs.head()

Clubs: 16
Club-sensor pairs after cleanup: 16
Unique nearby sensors after cleanup: 10


,club_name,rating,review_count,sensor_id,closest_sensor_description_manual,log_reviews,rating_x_log_reviews
0,Melbourne Club,4.5,53,18,Collins Place North,3.988984,17.950428
1,Athenaeum Club,4.6,71,39,Alfred Place,4.276666,19.672664
2,Alexandra Club,4.8,17,39,Alfred Place,2.890372,13.873784
3,Kelvin Club,4.4,142,39,Alfred Place,4.962845,21.836516
4,The Charlton Club,4.0,2455,63,155-161 Russell Street; 231 Bourke Street,7.806289,31.225157


In [7]:
# ------------------------------------------------------------
# Sensor metadata
# ------------------------------------------------------------

sensor_metadata = (
    df
    .sort_values("date")
    .groupby("sensor_id", as_index=False)
    .agg(
        sensor_label=("sensor_label", "last"),
        sensor_name=("sensor_name", "last"),
        lat=("lat", "last"),
        lon=("lon", "last"),
        status=("status", "last"),
        location_type=("location_type", "last"),
        note=("note", "last"),
    )
)


# Check researched sensors exist
missing_sensor_ids = sorted(set(club_sensor_ids) - set(sensor_metadata["sensor_id"]))

if missing_sensor_ids:
    print("Warning: these club sensors were not found in the pedestrian data:")
    print(missing_sensor_ids)
else:
    print("All researched club sensors were found.")


def weighted_average(values, weights):
    values = pd.Series(values)
    weights = pd.Series(weights)

    if weights.sum() == 0:
        return np.nan

    return np.average(values, weights=weights)


club_sensor_summary = (
    club_sensor_pairs
    .groupby("sensor_id")
    .apply(
        lambda g: pd.Series({
            "nearby_club_count": g["club_name"].nunique(),
            "nearby_clubs": "; ".join(
                g.sort_values("review_count", ascending=False)["club_name"]
            ),
            "total_reviews_nearby": g["review_count"].sum(),
            "max_reviews_single_club": g["review_count"].max(),
            "mean_club_rating": g["rating"].mean(),
            "review_weighted_rating": weighted_average(g["rating"], g["review_count"]),
            "sum_log_reviews": g["log_reviews"].sum(),
            "sum_rating_x_log_reviews": g["rating_x_log_reviews"].sum(),
            "manual_sensor_description": "; ".join(
                g["closest_sensor_description_manual"].unique()
            ),
        })
    )
    .reset_index()
    .merge(sensor_metadata, on="sensor_id", how="left")
)

club_sensor_summary.to_csv(TABLE_DIR / "club_sensor_popularity_summary.csv", index=False)


All researched club sensors were found.


In [8]:
# ------------------------------------------------------------
# Main filtered datasets
# ------------------------------------------------------------

club_df = df[df["sensor_id"].isin(club_sensor_ids)].copy()

club_nightlife = club_df[
    club_df["hour"].isin(nightlife_hour_order)
].copy()

club_weekend_nightlife = club_nightlife[
    club_nightlife["nightlife_day_type"] == "Weekend night"
].copy()

club_weekend_late = club_nightlife[
    (club_nightlife["nightlife_day_type"] == "Weekend night") &
    (club_nightlife["period"] == "Late night, 22:00-02:59")
].copy()

club_weekday_late = club_nightlife[
    (club_nightlife["nightlife_day_type"] == "Weekday night") &
    (club_nightlife["period"] == "Late night, 22:00-02:59")
].copy()


In [ ]:
# ------------------------------------------------------------
# Sensor-level pedestrian metrics
# ------------------------------------------------------------

sensor_all_metrics = (
    df
    .groupby("sensor_id", as_index=False)
    .agg(
        all_hours_avg=("pedestrian_count", "mean"),
        all_hours_total=("pedestrian_count", "sum"),
        observed_hours=("pedestrian_count", "size"),
    )
)

sensor_weekend_nightlife = (
    df[df["is_nightlife_hour"] & df["is_weekend_night"]]
    .groupby("sensor_id", as_index=False)
    .agg(
        weekend_nightlife_avg=("pedestrian_count", "mean"),
        weekend_nightlife_total=("pedestrian_count", "sum"),
        weekend_nightlife_hours=("pedestrian_count", "size"),
    )
)

sensor_weekday_nightlife = (
    df[df["is_nightlife_hour"] & df["is_weekday_night"]]
    .groupby("sensor_id", as_index=False)
    .agg(
        weekday_nightlife_avg=("pedestrian_count", "mean"),
        weekday_nightlife_total=("pedestrian_count", "sum"),
        weekday_nightlife_hours=("pedestrian_count", "size"),
    )
)

sensor_weekend_late = (
    df[df["is_late_night"] & df["is_weekend_night"]]
    .groupby("sensor_id", as_index=False)
    .agg(
        weekend_late_avg=("pedestrian_count", "mean"),
        weekend_late_total=("pedestrian_count", "sum"),
        weekend_late_hours=("pedestrian_count", "size"),
    )
)

sensor_weekday_late = (
    df[df["is_late_night"] & df["is_weekday_night"]]
    .groupby("sensor_id", as_index=False)
    .agg(
        weekday_late_avg=("pedestrian_count", "mean"),
        weekday_late_total=("pedestrian_count", "sum"),
        weekday_late_hours=("pedestrian_count", "size"),
    )
)

sensor_weekend_evening = (
    df[df["is_evening"] & df["is_weekend_night"]]
    .groupby("sensor_id", as_index=False)
    .agg(
        weekend_evening_avg=("pedestrian_count", "mean"),
        weekend_evening_total=("pedestrian_count", "sum"),
        weekend_evening_hours=("pedestrian_count", "size"),
    )
)


# Hourly peak metric
sensor_hourly_peak = (
    df[df["is_nightlife_hour"] & df["is_weekend_night"]]
    .groupby(["sensor_id", "hour"], as_index=False)
    .agg(avg_count=("pedestrian_count", "mean"))
    .sort_values(["sensor_id", "avg_count"], ascending=[True, False])
    .groupby("sensor_id", as_index=False)
    .first()
    .rename(columns={
        "hour": "best_weekend_nightlife_hour",
        "avg_count": "best_hour_avg_count"
    })
)


sensor_activity = (
    sensor_metadata
    .merge(sensor_all_metrics, on="sensor_id", how="left")
    .merge(sensor_weekend_nightlife, on="sensor_id", how="left")
    .merge(sensor_weekday_nightlife, on="sensor_id", how="left")
    .merge(sensor_weekend_late, on="sensor_id", how="left")
    .merge(sensor_weekday_late, on="sensor_id", how="left")
    .merge(sensor_weekend_evening, on="sensor_id", how="left")
    .merge(sensor_hourly_peak, on="sensor_id", how="left")
)

sensor_activity["weekend_late_vs_weekday_late_lift"] = (
    sensor_activity["weekend_late_avg"] / sensor_activity["weekday_late_avg"]
)

sensor_activity["weekend_nightlife_vs_weekday_nightlife_lift"] = (
    sensor_activity["weekend_nightlife_avg"] / sensor_activity["weekday_nightlife_avg"]
)


# ------------------------------------------------------------
# Join club popularity and pedestrian activity
# ------------------------------------------------------------

story_places = (
    club_sensor_summary
    .merge(
        sensor_activity.drop(
            columns=[
                "sensor_label",
                "sensor_name",
                "lat",
                "lon",
                "status",
                "location_type",
                "note"
            ],
            errors="ignore"
        ),
        on="sensor_id",
        how="left"
    )
)


# ------------------------------------------------------------
# Create score components
# ------------------------------------------------------------
# Percentile ranks make the score robust to extreme values.
# Higher score = stronger nightclubbing recommendation.

def percentile_score(series):
    return series.rank(pct=True).fillna(0)


story_places["pedestrian_intensity_score"] = percentile_score(
    story_places["weekend_late_avg"]
)

story_places["club_popularity_score"] = percentile_score(
    np.log1p(story_places["total_reviews_nearby"])
)

story_places["weekend_lift_score"] = percentile_score(
    story_places["weekend_late_vs_weekday_late_lift"].replace([np.inf, -np.inf], np.nan)
)

story_places["rating_score"] = story_places["review_weighted_rating"] / 5

story_places["hourly_peak_score"] = percentile_score(
    story_places["best_hour_avg_count"]
)


# Score weights
# You can adjust these if your story wants to prioritise a different idea of "best".
story_places["nightclubbing_score"] = (
    0.45 * story_places["pedestrian_intensity_score"] +
    0.25 * story_places["club_popularity_score"] +
    0.15 * story_places["weekend_lift_score"] +
    0.10 * story_places["rating_score"] +
    0.05 * story_places["hourly_peak_score"]
)

story_places["nightclubbing_score_100"] = 100 * story_places["nightclubbing_score"]

story_places.to_csv(TABLE_DIR / "best_nightclubbing_places_ranked.csv", index=False)

story_places[
    [
        "sensor_id",
        "sensor_label",
        "nearby_clubs",
        "nearby_club_count",
        "total_reviews_nearby",
        "review_weighted_rating",
        "weekend_late_avg",
        "weekday_late_avg",
        "weekend_late_vs_weekday_late_lift",
        "best_weekend_nightlife_hour",
        "nightclubbing_score_100"
    ]
]